# 第四步：RAG 检索与可视化

基于前一步生成的向量库和样本，随机选取查询序列进行相似检索，并绘制 TopK 对比图。

## 1. 参数与路径
按需修改：
- `VECTOR_DB_DIR`：向量库目录（query/target 可任选其一）
- `SAMPLES_DIR`：与向量库对应的样本目录
- `QUERY_NPZ`：查询样本来源
- `POOLING`：mean/last/max/weighted_mean
- `CHRONOS_MODEL_PATH`:时序模型参数

下载地址：https://modelscope.cn/models/better464/times


In [1]:
from pathlib import Path

INPUT_LEN = 168

VECTOR_DB_DIR = Path('./shaanxi_cities_temperature/vector_db_w192_s6/query')
SAMPLES_DIR   = Path('./shaanxi_cities_temperature/samples_w192_s6/query')
QUERY_NPZ     = Path('./shaanxi_cities_temperature/samples_w192_s6/query/西安_samples_w192_s6.npz')
OUT_DIR       = Path('./shaanxi_cities_temperature/results_demo')
POOLING       = 'last'
CHRONOS_MODEL_PATH = 'chronos'
TOPK = 3
NUM_QUERIES = 3
SEED = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('VECTOR_DB_DIR =', VECTOR_DB_DIR)
print('SAMPLES_DIR   =', SAMPLES_DIR)
print('QUERY_NPZ     =', QUERY_NPZ)
print('OUT_DIR       =', OUT_DIR)

VECTOR_DB_DIR = shaanxi_cities_temperature/vector_db_w192_s6/query
SAMPLES_DIR   = shaanxi_cities_temperature/samples_w192_s6/query
QUERY_NPZ     = shaanxi_cities_temperature/samples_w192_s6/query/西安_samples_w192_s6.npz
OUT_DIR       = shaanxi_cities_temperature/results_demo


## 2. RAG 类

In [2]:
import json
import pickle
from typing import Dict, List, Optional
import numpy as np
import torch
import faiss
import matplotlib.pyplot as plt
from chronos import ChronosPipeline

class CityTempRAG:
    def __init__(self, vector_db_dir: Path, samples_dir: Path, chronos_model_path: str, pooling: str = 'mean', device: Optional[str] = None):
        self.vector_db_dir = Path(vector_db_dir)
        self.samples_dir = Path(samples_dir)
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.pooling = pooling
        self.chronos_model_path = chronos_model_path
        self._load_vector_db()
        self._init_chronos()

    def _load_vector_db(self):
        self.embeddings = np.load(self.vector_db_dir / 'embeddings.npy')
        with open(self.vector_db_dir / 'id_mapping.pkl', 'rb') as f:
            id_map = pickle.load(f)
            self.sample_id_to_idx = id_map['sample_id_to_idx']
            self.idx_to_sample_id = id_map['idx_to_sample_id']
        with open(self.vector_db_dir / 'meta.json', 'r') as f:
            self.meta: List[Dict] = json.load(f)
        index_path = self.vector_db_dir / 'faiss_index.bin'
        if index_path.exists():
            self.index = faiss.read_index(str(index_path))
        else:
            dim = self.embeddings.shape[1]
            self.index = faiss.IndexFlatIP(dim)
            self.index.add(self.embeddings.astype(np.float32))
        self.emb_normalized = abs(np.linalg.norm(self.embeddings[0]) - 1.0) < 1e-3
        print(f"Loaded vector DB: {len(self.embeddings)} vectors, dim={self.embeddings.shape[1]}")

    def _init_chronos(self):
        print(f"Loading Chronos on {self.device} ...")
        self.chronos = ChronosPipeline.from_pretrained(
            self.chronos_model_path,
            device_map=self.device,
            dtype=torch.float32,
        )
        print('Chronos loaded')

    def _pool(self, embeds: torch.Tensor) -> torch.Tensor:
        if self.pooling == 'mean':
            return embeds.mean(dim=1)
        if self.pooling == 'last':
            return embeds[:, -1, :]
        if self.pooling == 'max':
            return embeds.max(dim=1)[0]
        if self.pooling == 'weighted_mean':
            seq_len = embeds.shape[1]
            w = torch.arange(1, seq_len + 1, device=embeds.device, dtype=torch.float32)
            w = w / w.sum()
            return (embeds * w.view(1, -1, 1)).sum(dim=1)
        raise ValueError(f'Unknown pooling {self.pooling}')

    def embed_query(self, series: np.ndarray) -> np.ndarray:
        ctx = torch.tensor(series[-INPUT_LEN:], dtype=torch.float32)
        with torch.no_grad():
            embeds_raw, _ = self.chronos.embed(ctx)
            pooled = self._pool(embeds_raw)
            q = pooled.cpu().numpy().reshape(1, -1)
        if self.emb_normalized:
            q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-8)
        return q.astype(np.float32)

    def search(self, series: np.ndarray, topk: int = 3):
        q = self.embed_query(series)
        scores, idxs = self.index.search(q, topk)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            sample_id = self.idx_to_sample_id[idx]
            meta = next(m for m in self.meta if m['sample_id'] == sample_id)
            results.append({'sample_id': sample_id, 'score': float(score), 'meta': meta})
        return results

    def fetch_sample(self, meta: Dict):
        npz_path = Path(meta['npz_path'])
        if not npz_path.is_absolute():
            npz_path = self.samples_dir / npz_path.name
        data = np.load(npz_path)
        idx = meta['idx_in_city']
        inp = data['input'][idx]
        tgt = data['target'][idx]
        return inp, tgt

/usr/local/miniconda3/envs/timerag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 绘图函数

In [3]:
import matplotlib.pyplot as plt
import numpy as np

COLORS = ['red','green','orange','purple','brown']

def plot_topk(query_inp, query_tgt, results, rag: CityTempRAG, out_path: Path):
    plt.figure(figsize=(10,6))
    query_seq = np.concatenate([query_inp, query_tgt])
    q_mean = query_seq.mean()
    q_plot = query_seq / q_mean if q_mean != 0 else query_seq
    plt.plot(np.arange(len(query_seq)), q_plot, label='query (input+target, /mean)', color='blue', linewidth=2)
    plt.axvline(x=INPUT_LEN-1, color='gray', linestyle='--', linewidth=1)

    for i, res in enumerate(results):
        inp, tgt = rag.fetch_sample(res['meta'])
        seq = np.concatenate([inp, tgt])
        s_mean = seq.mean()
        seq_plot = seq / s_mean if s_mean != 0 else seq
        plt.plot(np.arange(len(seq)), seq_plot, label=f"{i+1}:{res['sample_id']} (score {res['score']:.4f})", color=COLORS[i % len(COLORS)], alpha=0.85)

    plt.xlabel('time step'); plt.ylabel('value (per-series normalized)')
    plt.title(f'Top-{len(results)} similar sequences')
    plt.legend(); plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150); plt.close()
    print('saved:', out_path)

## 4. 运行检索与可视化
随机抽取 `NUM_QUERIES` 条样本，检索 TopK 并保存图像到 `OUT_DIR`。

In [4]:
import numpy as np
from pathlib import Path

rng = np.random.default_rng(SEED)
rag = CityTempRAG(
    vector_db_dir=VECTOR_DB_DIR,
    samples_dir=SAMPLES_DIR,
    pooling=POOLING,
    chronos_model_path=CHRONOS_MODEL_PATH,
)

query_data = np.load(QUERY_NPZ)
inputs, targets = query_data['input'], query_data['target']
num = min(NUM_QUERIES, len(inputs))
idxs = rng.choice(len(inputs), size=num, replace=False)
print('chosen idx:', idxs)

for i, idx in enumerate(idxs, 1):
    q_inp = inputs[idx]
    q_tgt = targets[idx]
    results = rag.search(q_inp, topk=TOPK)
    print(f"Query #{i} idx={idx} top{TOPK}:")
    for r in results:
        print(f"  {r['sample_id']}  score={r['score']:.4f}")
    out_path = OUT_DIR / f"query{idx}_top{TOPK}.png"
    plot_topk(q_inp, q_tgt, results, rag, out_path)

Loaded vector DB: 481810 vectors, dim=1024
Loading Chronos on cuda ...
Chronos loaded
chosen idx: [31537  4300 37289]
Query #1 idx=31537 top3:
  xian_2012080206  score=1.0000
  xian_2023080806  score=0.9889
  xian_2019080906  score=0.9862
saved: shaanxi_cities_temperature/results_demo/query31537_top3.png
Query #2 idx=4300 top3:
  xian_1993121100  score=1.0000
  lanzhou_2004122200  score=0.9878
  lanzhou_2020120900  score=0.9854
saved: shaanxi_cities_temperature/results_demo/query4300_top3.png
Query #3 idx=37289 top3:
  xian_2016071006  score=1.0000
  xian_2019080606  score=0.9878
  xian_2023082306  score=0.9872
saved: shaanxi_cities_temperature/results_demo/query37289_top3.png


## 5. 查看输出文件

In [5]:
import glob
print('结果图：')
for f in sorted(glob.glob(str(OUT_DIR / '*.png'))):
    print(f)

结果图：
shaanxi_cities_temperature/results_demo/query31537_top3.png
shaanxi_cities_temperature/results_demo/query37289_top3.png
shaanxi_cities_temperature/results_demo/query4300_top3.png
